<a href="https://colab.research.google.com/github/rameshaditya-me/Easy-Classical-ML-DL/blob/main/src/notebooks/generalized-linear-models/generalized-linear-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generalized Linear Models
---

**Generalized Linear Models (GLMs)** extend ordinary linear regression to many kinds of response variable. The same linear structure in the predictors can model continuous, discrete, binary, ordinal, categorical, and count outcomes — the choice of **link function** determines how $\beta^\top X$ connects to the distribution of $Y$.

The key idea is twofold. First, a link $g$ maps the linear predictor $\beta^\top X$ to a parameter of the outcome distribution (e.g. the mean). Second, the variance of each observation may depend on that predicted value — heteroscedasticity is built into the model rather than treated as a nuisance.

A GLM has three parts:

1. **Random component.** The outcome $Y$ is drawn from an **exponential-family** distribution (Gaussian, Bernoulli, Poisson, etc.).

2. **Systematic component.** One or more covariates $X_0, \ldots, X_d$ enter linearly through $\beta^\top X$.

3. **Link function.** A function $g$ connects the systematic component to the mean (or natural parameter) of the random component: $g(\mathbb{E}[Y \mid X]) = \beta^\top X$. A link function is called a canonical-link-function when it is directly equal to the natural parameter $\eta{\theta}$ of the exponential family.

### Exponential Family of Probability Distributions
---
Any probability distribution that can be written in the following parametric form belongs to the **exponential family**:

$$
p_{\theta}(y) = h(y)\,\exp\!\left(\eta(\theta)\,T(y) - A(\theta)\right)
$$

where $h(y)$ is the **base measure**, $\eta(\theta)$ is the **natural parameter**, $T(y)$ is the **sufficient statistic**, and $A(\theta)$ is the **log-partition function**.

#### Gaussian with unknown mean and known variance

$$
p(y) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(y-\mu)^2}{2\sigma^2}\right)
$$

Expanding $(y-\mu)^2$ and matching the exponential-family form:

$$
p(y) =
\underset{\text{base measure}}{\overbrace{
  \frac{1}{\sqrt{2\pi\sigma^2}}\,
  \exp\!\left(-\dfrac{y^2}{2\sigma^2}\right)
}^{h(y)}}\;
\exp\!\left(
  \underset{\text{natural parameter}}{\overbrace{\dfrac{\mu}{\sigma^2}}^{\eta(\theta)}}\,
  \underset{\text{sufficient statistic}}{\overbrace{y}^{T(y)}}
  -
  \underset{\text{log-partition}}{\overbrace{\dfrac{\mu^2}{2\sigma^2}}^{A(\theta)}}
\right)
$$

Here $\theta = \mu$, so $\eta(\mu) = \mu/\sigma^2$ and $T(y) = y$.

#### Binomial distribution with a known number of trials $n$

Let $Y \in \{0, 1, \ldots, n\}$ count the number of successes in $n$ independent trials, each with success probability $p$:

$$
P(y) = \binom{n}{y}\, p^{y}(1-p)^{n-y}
$$

Rewriting in log form and collecting terms in $y$:

$$
P(y) = \binom{n}{y}\exp\!\left(y\log\frac{p}{1-p} + n\log(1-p)\right)
$$

Matching the exponential-family form $p_{\theta}(y) = h(y)\exp(\eta(\theta)T(y) - A(\theta))$:

$$
P(y) =
\underset{\text{base measure}}{\overbrace{
  \binom{n}{y}
}^{h(y)}}\;
\exp\!\left(
  \underset{\text{natural parameter}}{\overbrace{\log\!\left(\dfrac{p}{1-p}\right)}^{\eta(\theta)}}\,
  \underset{\text{sufficient statistic}}{\overbrace{y}^{T(y)}}
  -
  \underset{\text{log-partition}}{\overbrace{\left(-n\log(1-p)\right)}^{A(\theta)}}
\right)
$$

Here $\theta = p$, so $\eta(p) = \log\!\big(p/(1-p)\big)$ (the **log-odds**) and $T(y) = y$. Equivalently, writing $A$ in terms of the natural parameter alone:

$$
A(\eta) = n\log\!\left(1 + e^{\eta}\right)
$$

The **Bernoulli** distribution is the special case $n = 1$; logistic regression uses this exponential-family form with the logit link $\eta = \beta^\top x$.

### Choosing a GLM
---

A GLM is specified by three linked choices: the **type of response** you observe, the **exponential-family distribution** that models $Y$, and the **link function** $g$ that connects the linear predictor $\beta^\top X$ to a parameter of that distribution.

Ordinary linear regression is the GLM with $Y \sim \mathcal{N}(\mu, \sigma^2)$ and the **identity** link $g(\mu) = \mu$. Logistic regression is the GLM with $Y \sim \text{Bernoulli}(p)$ and the **logit** link $g(p) = \log\!\big(p/(1-p)\big)$. Every other exponential-family member — Poisson, Gamma, Negative Binomial, and so on — defines its own regression model once paired with a link, usually the **canonical link** that sets the natural parameter $\eta$ equal to $\beta^\top X$.

**Step1:** Given a dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$, the first step in fitting a GLM is to identify the response type and select the matching family and link:

| Nature of data | Target family | Canonical link $g(\mu)$ |
| --- | --- | --- |
| Continuous in $\mathbb{R}$ | $Y \sim \mathcal{N}(\mu, \sigma^2)$ | Identity: $\beta^\top X = \mu$ |
| Binary in $\{0, 1\}$ | $Y \sim \text{Bernoulli}(p)$ | Logit: $\beta^\top X = \log\!\dfrac{p}{1-p}$ |
| Counts in $\{0, 1, 2, \ldots\}$ | $Y \sim \text{Poisson}(\lambda)$ | Log: $\beta^\top X = \log \lambda$ |
| Counts with overdispersion | $Y \sim \text{NegBinomial}(r, p)$ | Log: $\beta^\top X = \log \mu$ |
| Positive continuous | $Y \sim \text{Gamma}(\alpha, \beta)$ | Inverse: $\beta^\top X = 1/\mu$ |
| Positive continuous (skewed) | $Y \sim \text{Exponential}(\lambda)$ | Log: $\beta^\top X = \log \lambda$ |

**Step 2:** Estimate the regression coefficients $\beta$ by **maximum likelihood** — find the $\beta$ that maximizes the likelihood of the observed $y_i$ under the chosen exponential-family distribution (equivalently, minimizes the deviance).

**Step 3:** To predict at a new input $x$, evaluate $\hat\beta^\top x$ to obtain the natural parameter $\hat\eta$, which fully specifies the fitted distribution. In practice we usually report the predicted mean $\mathbb{E}[Y \mid x]$, but the distributional form also lets us construct **prediction intervals** or **confidence bands** around $\hat{y}$.

### Confidence Intervals
---

Once a GLM is fitted, a point prediction at a new input $x$ is the estimated mean response

$$
\hat\mu = g^{-1}(\hat\beta^\top x),
$$

where $g$ is the link function. To quantify uncertainty, we distinguish two targets:

- A **confidence interval (CI) for the mean** $\mathbb{E}[Y \mid x]$ — how precisely we have estimated the fitted response at $x$.
- A **prediction interval (PI) for a new observation** $Y_{\text{new}}$ — the range a future outcome at $x$ is expected to fall in, including irreducible randomness in $Y$ itself.

To build either interval we need (i) a point estimate $\hat\beta$, (ii) a measure of how uncertain that estimate is, and (iii) a rule for turning estimate $\pm$ uncertainty into an interval. The three ideas below supply exactly that.

#### Fisher information

Given a dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$, the GLM log-likelihood aggregates over **all $n$ observations**:

$$
\ell(\beta) = \log p(y \mid X, \beta) = \sum_{i=1}^n \log p(y_i \mid x_i, \beta).
$$

The **Fisher information matrix** measures how sharply this $n$-point log-likelihood curves around $\beta$ — equivalently, how much information the full dataset carries about the coefficients:

$$
\mathcal{I}(\beta) = -\mathbb{E}\!\left[\frac{\partial^2 \ell(\beta)}{\partial \beta \,\partial \beta^\top}\right] = -\sum_{i=1}^n \mathbb{E}\!\left[\frac{\partial^2 \log p(y_i \mid x_i, \beta)}{\partial \beta \,\partial \beta^\top}\right].
$$

The expectation is over the random responses $y_i$; each new datapoint adds a term, so $\mathcal{I}(\beta)$ grows with $n$.

In practice we use the **observed (empirical) Fisher information** — the negative Hessian of $\ell$ evaluated at the MLE, with no expectation:

$$
\widehat{\mathcal{I}}(\hat\beta) = -\left.\frac{\partial^2 \ell(\beta)}{\partial \beta \,\partial \beta^\top}\right|_{\beta = \hat\beta}.
$$

Large curvature of $\ell$ (large $\mathcal{I}$ or $\widehat{\mathcal{I}}$) means the likelihood is sharply peaked and $\beta$ is tightly pinned down; flat curvature means more uncertainty. For large $n$, the MLE satisfies the approximate normality result

$$
\hat\beta \;\approx\; \mathcal{N}\!\left(\beta,\; \mathcal{I}(\hat\beta)^{-1}\right) \;\approx\; \mathcal{N}\!\left(\beta,\; \widehat{\mathcal{I}}(\hat\beta)^{-1}\right),
$$



#### Fisher information grows with sample size

Consider a Gaussian GLM with $\beta = (\beta_0, \beta_1)^\top$, identity link, and known $\sigma^2$. For design matrix $X \in \mathbb{R}^{n \times 2}$,

$$
\mathcal{I}(\beta) = \frac{1}{\sigma^2}\, X^\top X,
$$

so each new observation adds curvature to $\ell(\beta)$. The animation below plots **precomputed** log-likelihood contours on the $(\beta_0, \beta_1)$ plane as $n$ grows. Sharper, tighter contours correspond to larger $\det(\mathcal{I})$ — the likelihood concentrates more tightly around $\hat\beta$.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.ioff()  # prevent inline backend from also rendering a static figure

# --- fixed data-generating process ---
rng = np.random.default_rng(42)
TRUE_BETA = np.array([1.0, 2.0])  # (intercept, slope)
SIGMA = 1.0
MAX_N = 200

x_full = rng.uniform(-2, 2, MAX_N)
X_full = np.column_stack([np.ones(MAX_N), x_full])
y_full = X_full @ TRUE_BETA + rng.normal(0, SIGMA, MAX_N)

# --- β grid for contour plots ---
b0_grid = np.linspace(-1.0, 3.0, 100)
b1_grid = np.linspace(0.0, 4.0, 100)
B0, B1 = np.meshgrid(b0_grid, b1_grid)


def log_likelihood_grid(B0, B1, X, y, sigma):
    """Gaussian log-likelihood evaluated on a 2-D β grid."""
    ll = np.zeros_like(B0, dtype=float)
    for i in range(len(y)):
        mu = B0 + B1 * X[i, 1]
        ll += -0.5 * ((y[i] - mu) / sigma) ** 2
    ll -= len(y) * np.log(sigma * np.sqrt(2 * np.pi))
    return ll


def fisher_information(X, sigma):
    return (X.T @ X) / sigma**2


# --- precompute contours and Fisher information for each sample size ---
SAMPLE_SIZES = [5, 10, 20, 40, 80, 120, 160, MAX_N]

precomputed = {}
for n in SAMPLE_SIZES:
    X_n = X_full[:n]
    y_n = y_full[:n]
    Z = log_likelihood_grid(B0, B1, X_n, y_n, SIGMA)
    beta_hat = np.linalg.lstsq(X_n, y_n, rcond=None)[0]
    I = fisher_information(X_n, SIGMA)
    precomputed[n] = {
        "Z_rel": Z - Z.max(),  # peak at 0 for comparable contour levels
        "beta_hat": beta_hat,
        "det_I": np.linalg.det(I),
    }

Z_MIN, Z_MAX = -35, 0
extent = [b0_grid[0], b0_grid[-1], b1_grid[0], b1_grid[-1]]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
first = precomputed[SAMPLE_SIZES[0]]
im = ax.imshow(
    first["Z_rel"],
    origin="lower",
    extent=extent,
    cmap="viridis",
    vmin=Z_MIN,
    vmax=Z_MAX,
    aspect="auto",
    interpolation="bilinear",
)
fig.colorbar(im, ax=ax, label=r"$\ell(\beta) - \max_\beta \ell(\beta)$")

true_beta_dot, = ax.plot(
    [TRUE_BETA[0]], [TRUE_BETA[1]], "r*", markersize=15, label=r"true $\beta$", zorder=5,
)
mle_dot, = ax.plot(
    [first["beta_hat"][0]], [first["beta_hat"][1]],
    "o", color="white", markeredgecolor="black", markersize=9,
    label=rf"MLE $\hat\beta$", zorder=5,
)

text_box = ax.text(
    0.02, 0.98, "", transform=ax.transAxes, va="top", fontsize=10,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
)

ax.set_xlabel(r"$\beta_0$ (intercept)")
ax.set_ylabel(r"$\beta_1$ (slope)")
ax.legend(loc="upper right")


def _frame_text(n, data):
    return (
        f"n = {n} samples\n"
        f"$\\det(\\mathcal{{I}}) = {data['det_I']:.1f}$\n"
        f"$\\hat\\beta = ({data['beta_hat'][0]:.2f},\\; {data['beta_hat'][1]:.2f})$"
    )


ax.set_title(rf"Log-likelihood contours  ($n = {SAMPLE_SIZES[0]}$)")
text_box.set_text(_frame_text(SAMPLE_SIZES[0], first))


def update(frame):
    n = SAMPLE_SIZES[frame]
    data = precomputed[n]
    im.set_data(data["Z_rel"])
    mle_dot.set_data([data["beta_hat"][0]], [data["beta_hat"][1]])
    ax.set_title(rf"Log-likelihood contours  ($n = {n}$)")
    text_box.set_text(_frame_text(n, data))
    return [im, mle_dot, text_box]


anim = animation.FuncAnimation(
    fig, update, frames=len(SAMPLE_SIZES), blit=False, interval=900,
)
anim.event_source.stop()
html = anim.to_jshtml()
plt.close(fig)
display(HTML(html))


so $\widehat{\mathrm{Cov}}(\hat\beta) \approx \mathcal{I}(\hat\beta)^{-1}$ is the standard large-sample covariance of $\hat\beta$.

#### From curvature to IRLS

We now know how to *measure* uncertainty at the MLE — but how do we actually *find* $\hat\beta$ for a GLM? The log-likelihood $\ell(\beta)$ is generally nonlinear in $\beta$ (because of the link $g$), so there is no closed-form solution. The natural starting point is to climb the log-likelihood surface.

**Gradient ascent with a fixed learning rate.** The simplest update is

$$
\beta^{(t+1)} = \beta^{(t)} + \alpha\, \nabla_\beta \ell(\beta^{(t)}),
$$

where $\alpha > 0$ is a learning rate. This works in principle, but a single scalar $\alpha$ is awkward: directions where $\ell$ curves sharply (large Fisher information) can tolerate a large step, while flat directions need a small one. A step that is too large overshoots the peak; one that is too small crawls.

**Let curvature set the step size.** Near $\beta^{(t)}$, Taylor-expand $\ell$ to second order:

$$
\ell(\beta) \approx \ell(\beta^{(t)}) + \nabla_\beta \ell^\top (\beta - \beta^{(t)}) - \tfrac{1}{2}(\beta - \beta^{(t)})^\top H^{(t)} (\beta - \beta^{(t)}),
$$

where $H^{(t)} = -\nabla^2_\beta \ell(\beta^{(t)})$ is the **local curvature** (the observed Fisher information at $\beta^{(t)}$). To maximise this quadratic approximation, set $\delta = \beta - \beta^{(t)}$ and write the approximation as a function of $\delta$:

$$
Q(\delta) = \ell(\beta^{(t)}) + \nabla_\beta \ell(\beta^{(t)})^\top \delta - \tfrac{1}{2}\, \delta^\top H^{(t)} \delta.
$$

Differentiate with respect to $\delta$ and set to zero (the peak of a concave quadratic):

$$
\nabla_\delta Q(\delta) = \nabla_\beta \ell(\beta^{(t)}) - H^{(t)} \delta = 0
\quad\Longrightarrow\quad
\delta = \big(H^{(t)}\big)^{-1} \nabla_\beta \ell(\beta^{(t)}).
$$

Substituting $\beta^{(t+1)} = \beta^{(t)} + \delta$ gives **Newton's method**:

$$
\beta^{(t+1)} = \beta^{(t)} + \big(H^{(t)}\big)^{-1} \nabla_\beta \ell(\beta^{(t)}).
$$

Read this as gradient ascent with a *matrix-valued* learning rate $(H^{(t)})^{-1}$ that adapts to the shape of $\ell$ at each step. **Thus, steep directions get a shorter effective step, flat directions a longer one**.

**Where $W$ enters.** For an exponential-family GLM, this curvature decomposes across observations. Each datapoint contributes a scalar weight $w_i$ — how sharply $\log p(y_i \mid x_i, \beta)$ curves with respect to the linear predictor $\eta_i = \beta^\top x_i$. Stacking these gives a diagonal weight matrix

$$
W = \mathrm{diag}(w_1, \ldots, w_n), \qquad w_i = \frac{1}{\phi \cdot v(\mu_i) \cdot g'(\mu_i)^2},
$$

where $\mu_i = g^{-1}(\beta^\top x_i)$, $v(\mu)$ is the variance function of the family, and $\phi$ is the dispersion parameter. This is **not** a coefficient vector — it is a per-observation learning-rate multiplier derived from local curvature. Ordinary least squares is the special case $W = I$.

The full Hessian then takes the familiar form

$$
H^{(t)} = \frac{1}{\phi}\, X^\top W X,
$$

which is exactly the observed Fisher information matrix from the previous section.

**IRLS = Newton rewritten as weighted regression.** The Newton step can be re-expressed as a weighted least-squares problem — that is what **iteratively reweighted least squares (IRLS)** computes. At each iteration, build a working response $z_i$ (a first-order correction to $\eta_i$) and solve for the $\beta$ that best fits $z$ under weights $w_i$:

1. $\beta \leftarrow$ initial guess

2. **repeat**
   - for each $i = 1, \ldots, n$:
     - $\eta_i \leftarrow \beta^\top x_i$
     - $\mu_i \leftarrow g^{-1}(\eta_i)$
     - $w_i \leftarrow \big(\phi \cdot v(\mu_i) \cdot g'(\mu_i)^2\big)^{-1}$
     - $z_i \leftarrow \eta_i + (y_i - \mu_i)\, g'(\mu_i)$
   - $W \leftarrow \mathrm{diag}(w_1, \ldots, w_n)$
   - $\beta_{\text{new}} \leftarrow (X^\top W X)^{-1} X^\top W z$
   - **if** $\|\beta_{\text{new}} - \beta\| < \varepsilon$ **or** $|\ell(\beta_{\text{new}}) - \ell(\beta)| < \delta$: **break**
   - $\beta \leftarrow \beta_{\text{new}}$

3. **until** max iterations

4. **return** $\beta$

Because $w_i$ depends on $\mu_i$, which depends on $\beta$, the weights are recomputed every iteration. Convergence is declared when $\|\beta^{(t+1)} - \beta^{(t)}\| < \varepsilon$ or $|\ell(\beta^{(t+1)}) - \ell(\beta^{(t)})| < \delta$ (typical tolerances: $\varepsilon \sim 10^{-6}$, $\delta \sim 10^{-8}$).

**At convergence**, the curvature matrix $H$ at $\hat\beta$ gives both the Fisher information and the coefficient covariance:

$$
\widehat{\mathcal{I}}(\hat\beta) = \frac{1}{\phi}\, X^\top W X, \qquad \widehat{\mathrm{Cov}}(\hat\beta) = (X^\top W X)^{-1}\phi.
$$

Each new row of $X$ adds a rank-one term $w_i\, x_i x_i^\top$ to $X^\top W X$, which is why Fisher information — and the precision of $\hat\beta$ — grows with $n$.





#### Wald interval

Once we have an estimate $\hat\theta$ and a standard error $\mathrm{SE}(\hat\theta)$ that measures how much it can move, we can turn "estimate $\pm$ uncertainty" into a confidence interval. The **Wald interval** assumes $\hat\theta$ is approximately normal with this spread.

For a standard normal variable $Z$, the **Gaussian Q-function** is

$$
Q(x) = P(Z > x),
$$

the probability of landing in the upper tail beyond $x$. To build a $100(1-\alpha)\%$ interval, choose $q$ so that only a fraction $\alpha/2$ of the probability sits in **each** tail:

$$
Q(q) = \frac{\alpha}{2}.
$$

Then the Wald interval is simply

$$
\hat\theta \;\pm\; q \cdot \mathrm{SE}(\hat\theta).
$$

In words: start at $\hat\theta$, then walk $q$ standard errors in either direction. For a 95% interval ($\alpha = 0.05$), $Q(q) = 0.025$, giving $q \approx 1.96$. When an extra nuisance parameter such as $\sigma^2$ must also be estimated from the data (as in linear regression), the normal approximation is replaced by a $t$-distribution with the appropriate degrees of freedom.

---

**Putting it together.** IRLS gives us $\hat\beta$ and $\widehat{\mathrm{Cov}}(\hat\beta)$. To make a statement at a new input $x$, we first form the linear predictor $\hat\eta = \hat\beta^\top x$. Because $\hat\beta$ is uncertain, so is $\hat\eta$; the amount of spread is

$$
\mathrm{SE}(\hat\eta) = \sqrt{x^\top \widehat{\mathrm{Cov}}(\hat\beta)\, x}.
$$

Think of this as: "how much could $\hat\eta$ wiggle, given how uncertain each coordinate of $\hat\beta$ is, and how $x$ mixes those coordinates together?"

The general recipe is:

1. Build a Wald interval on the **link scale**: $\hat\eta \pm q \cdot \mathrm{SE}(\hat\eta)$, where $q$ satisfies $Q(q) = \alpha/2$.
2. Transform both endpoints through $g^{-1}$ to get a confidence interval for the mean $\mu$.
3. To predict a **new** observation $y_{\text{new}}$, add the observation-level variance $v(\mu)$ from the chosen family (e.g. $\sigma^2$ for Gaussian, $\mu$ for Poisson) on top of the uncertainty in step 1.

---


#### Example 3: IRLS logistic regression in practice

We fit a Bernoulli GLM with logit link using **IRLS from scratch** on a synthetic binary dataset (`make_classification` — the classification analogue of `make_regression`). The panels show the loss landscape with IRLS steps projected via PCA, a Wald confidence band for $P(y{=}1 \mid x)$, and the fitted logistic curve with classified datapoints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import norm
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

# --- synthetic binary data (2 features for visualization) ---
X_raw, y = make_classification(
    n_samples=200,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.4,
    random_state=42,
)
scaler = StandardScaler()
X_feat = scaler.fit_transform(X_raw)
X = np.column_stack([np.ones(len(y)), X_feat])  # intercept + 2 features
feature_names = ["feature 1", "feature 2"]


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def neg_log_likelihood(beta, X, y):
    mu = sigmoid(X @ beta)
    mu = np.clip(mu, 1e-12, 1 - 1e-12)
    return -np.sum(y * np.log(mu) + (1 - y) * np.log(1 - mu))


def grad_neg_log_likelihood(beta, X, y):
    mu = sigmoid(X @ beta)
    return -X.T @ (y - mu)


def irls_logistic(X, y, max_iter=50, tol=1e-8):
    """Bernoulli GLM with logit link, fit by IRLS."""
    beta = np.zeros(X.shape[1])
    history = [beta.copy()]

    for _ in range(max_iter):
        eta = X @ beta
        mu = sigmoid(eta)
        w = np.clip(mu * (1 - mu), 1e-8, None)
        z = eta + (y - mu) / w  # working response; g'(mu) = 1/(mu(1-mu))

        XtWX = X.T @ (w[:, None] * X)
        beta_new = np.linalg.solve(XtWX, X.T @ (w * z))
        history.append(beta_new.copy())

        if np.linalg.norm(beta_new - beta) < tol:
            beta = beta_new
            break
        beta = beta_new

    cov_beta = np.linalg.inv(XtWX)
    return beta, np.array(history), cov_beta


beta_hat, history, cov_beta = irls_logistic(X, y)

# --- PCA basis for projecting 3-D β space to 2-D ---
pca = PCA(n_components=2)
pca.fit(history)

# Loss surface samples on a 3-D β grid, projected to PCA coordinates
pad = 1.5
b_ranges = [
    np.linspace(history[:, i].min() - pad, history[:, i].max() + pad, 40)
    for i in range(3)
]
B0, B1, B2 = np.meshgrid(*b_ranges, indexing="ij")
beta_grid = np.column_stack([B0.ravel(), B1.ravel(), B2.ravel()])
loss_grid = np.array([neg_log_likelihood(b, X, y) for b in beta_grid])
loss_rel = loss_grid - loss_grid.min()
beta_pca = pca.transform(beta_grid)

history_pca = pca.transform(history)
grad_pca = np.array([pca.components_ @ grad_neg_log_likelihood(b, X, y) for b in history])

# --- confidence band: vary feature 1, fix feature 2 at its median ---
x1_line = np.linspace(X_feat[:, 0].min() - 0.5, X_feat[:, 0].max() + 0.5, 120)
x2_fixed = np.median(X_feat[:, 1])
X_line = np.column_stack([np.ones_like(x1_line), x1_line, np.full_like(x1_line, x2_fixed)])

eta_line = X_line @ beta_hat
se_eta = np.sqrt(np.sum((X_line @ cov_beta) * X_line, axis=1))
q = norm.ppf(0.975)  # Q(q) = 0.025 for 95% Wald interval

p_hat = sigmoid(eta_line)
p_lo = sigmoid(eta_line - q * se_eta)
p_hi = sigmoid(eta_line + q * se_eta)

# --- figure ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel 1: loss contours + IRLS path in PCA space
ax = axes[0]
levels = np.linspace(0, np.percentile(loss_rel, 92), 12)
ax.tricontourf(
    beta_pca[:, 0], beta_pca[:, 1], loss_rel,
    levels=levels, cmap="viridis", alpha=0.9,
)
ax.tricontour(
    beta_pca[:, 0], beta_pca[:, 1], loss_rel,
    levels=levels, colors="k", linewidths=0.3, alpha=0.4,
)
ax.plot(history_pca[:, 0], history_pca[:, 1], "o-", color="white",
        markeredgecolor="black", markersize=4, lw=1.5, label="IRLS path")
beta_hat_pca = pca.transform(beta_hat.reshape(1, -1))[0]
ax.scatter(beta_hat_pca[0], beta_hat_pca[1], c="red", s=80,
           marker="*", zorder=5, label=r"$\hat\beta$ (MLE)")

step = max(1, len(history) // 8)
for t in range(0, len(history) - 1, step):
    ax.annotate(
        "", xy=(history_pca[t + 1, 0], history_pca[t + 1, 1]),
        xytext=(history_pca[t, 0], history_pca[t, 1]),
        arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.8),
    )
ax.quiver(
    history_pca[::step, 0], history_pca[::step, 1],
    -grad_pca[::step, 0], -grad_pca[::step, 1],
    angles="xy", scale_units="xy", scale=3.5,
    color="#e67e22", alpha=0.85, width=0.004, label="$-\\nabla \\ell$ (PCA)",
)
ax.set_xlabel("PC 1 of $\\beta$")
ax.set_ylabel("PC 2 of $\\beta$")
ax.set_title("Loss surface & IRLS steps (PCA)")
ax.legend(loc="upper right", fontsize=8)

# Panel 2: Wald confidence band for $p(y=1|x)$
ax = axes[1]
ax.fill_between(x1_line, p_lo, p_hi, alpha=0.25, color="#3498db", label="95% Wald CI")
ax.plot(x1_line, p_hat, color="#2c3e50", lw=2.5, label=r"$\hat p = \sigma(\hat\eta)$")
ax.axhline(0.5, color="gray", ls=":", lw=1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(f"{feature_names[0]} (standardized)")
ax.set_ylabel(r"$P(y=1 \mid x)$")
ax.set_title(f"Confidence band ($x_2$ fixed at {x2_fixed:.2f})")
ax.legend(loc="lower right", fontsize=8)

# Panel 3: datapoints, decision boundary, and logistic curve (inset)
ax = axes[2]
y_pred = (sigmoid(X @ beta_hat) >= 0.5).astype(int)
palette = {1: "#e74c3c", 0: "#3498db"}

for i in range(len(y)):
    correct = y_pred[i] == y[i]
    ax.scatter(
        X_feat[i, 0], X_feat[i, 1],
        c=palette[y[i]], s=55 if correct else 80,
        marker="o" if correct else "X",
        edgecolors="white", linewidths=0.5, alpha=0.85,
    )

x1_db = np.linspace(X_feat[:, 0].min() - 0.3, X_feat[:, 0].max() + 0.3, 100)
x2_db = -(beta_hat[0] + beta_hat[1] * x1_db) / beta_hat[2]
ax.plot(x1_db, x2_db, "k-", lw=2, label="decision boundary ($\\hat p=0.5$)")

axins = inset_axes(ax, width="42%", height="42%", loc="lower left", borderpad=2)
axins.fill_between(x1_line, p_lo, p_hi, alpha=0.2, color="#3498db")
axins.plot(x1_line, p_hat, color="#2c3e50", lw=1.8)
axins.axhline(0.5, color="gray", ls=":", lw=0.8)
axins.set_xlabel(r"$x_1$", fontsize=7)
axins.set_ylabel(r"$\hat p$", fontsize=7)
axins.tick_params(labelsize=6)
axins.set_ylim(-0.05, 1.05)
axins.set_title("logistic curve", fontsize=7)

ax.scatter([], [], c="#e74c3c", marker="o", label="class 1 (correct)")
ax.scatter([], [], c="#3498db", marker="o", label="class 0 (correct)")
ax.scatter([], [], c="gray", marker="X", label="misclassified")
ax.set_xlabel(f"{feature_names[0]} (standardized)")
ax.set_ylabel(f"{feature_names[1]} (standardized)")
ax.set_title("Classification in feature space")
ax.legend(loc="upper right", fontsize=7)

plt.tight_layout()
plt.show()

print(f"IRLS converged in {len(history) - 1} iterations")
print(f"beta_hat = {beta_hat.round(4)}")



#### Example 1: Linear regression (Gaussian GLM, identity link)

Model: $Y \mid x \sim \mathcal{N}(\mu, \sigma^2)$ with $\mu = \beta^\top x$.

**Point prediction:** $\hat y = \hat\beta^\top x$.

**95% CI for the mean** $\mathbb{E}[Y \mid x]$:

$$
\hat y \;\pm\; t_{n-p-1,\;\alpha/2}\;\hat\sigma\;\sqrt{x^\top (X^\top X)^{-1} x},
$$

where $\hat\sigma^2$ is the residual mean square, $p$ is the number of coefficients, and $t_{n-p-1,\;\alpha/2}$ accounts for estimating $\sigma^2$.

**95% prediction interval** for a new observation $Y_{\text{new}}$ at the same $x$:

$$
\hat y \;\pm\; t_{n-p-1,\;\alpha/2}\;\hat\sigma\;\sqrt{1 + x^\top (X^\top X)^{-1} x}.
$$

The extra $+1$ under the square root is the contribution of the observation noise $\sigma^2$ — the PI is always wider than the CI for the mean.

---

#### Example 2: Logistic regression (Bernoulli GLM, logit link)

Model: $Y \mid x \sim \text{Bernoulli}(p)$ with $\log\!\dfrac{p}{1-p} = \beta^\top x$.

**Point prediction:** $\hat p = \sigma(\hat\beta^\top x) = \dfrac{1}{1 + e^{-\hat\beta^\top x}}$.

Because $p$ is bounded in $[0,1]$, we form the interval on the **log-odds scale** $\hat\eta = \hat\beta^\top x$ and transform back:

$$
\mathrm{SE}(\hat\eta) = \sqrt{x^\top \widehat{\mathrm{Cov}}(\hat\beta)\, x},
$$

$$
\hat p \;\in\; \left(\sigma\!\big(\hat\eta - q\,\mathrm{SE}(\hat\eta)\big),\;\; \sigma\!\big(\hat\eta + q\,\mathrm{SE}(\hat\eta)\big)\right), \qquad Q(q) = \tfrac{\alpha}{2}.
$$

This is a **95% CI for the success probability** $\mathbb{E}[Y \mid x] = p$.

For a single Bernoulli outcome, $Y_{\text{new}} \in \{0, 1\}$, so a classical symmetric PI does not apply. Instead, the fitted $\hat p$ is the predictive probability $\mathbb{P}(Y_{\text{new}} = 1 \mid x)$, and uncertainty about that probability is captured by the CI above.